In [ ]:
!pip install -q transformers>=4.45.0 accelerate bitsandbytes peft trl \
    qwen-vl-utils pillow datasets wandb

In [ ]:
## To reduce the redundant compuation of the image embeddings, it can be cached once computed
import time
from collections import OrderedDict
class EmbeddingCache:
    def __init__(self, ttl_sec=60, max_size=64):
        self._store = OrderedDict()
        self._ttl = ttl_sec
        self._max = max_size


    def get(self, key):
        if key not in self._store:
            return None
        value, exp = self._store[key]
        if time.monotonic() > exp:
            del self._store[key]
            return None
        self._store.move_to_end(key)
        return value

    def set(self, key, value):
        self._store[key] = (value, time.monotonic() + self._ttl)
        self._store.move_to_end(key)
        if len(self._store) > self._max:
            self._store.popitem(last=False)

embed_cache = EmbeddingCache(ttl_sec=60)


In [ ]:
## Load QWEN model
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)


model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",    ## CPU/GPU
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)

In [ ]:
from PIL import Image
import requests
import os
from io import BytesIO
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive/')
def build_qwen_messages(image, question):

    content = []


    content.append({"type": "image", "image": image})

    content.append({"type": "text", "text": f"\nQuestion: {question}"})

    messages = [
        {
            "role": "system",
            "content": (
                "You are an autonomous driving assistant. "
                "You are given front view camera image"
                "Answer concisely based on what you observe."
            ),
        },
        {"role": "user", "content": content},
    ]
    return messages


In [ ]:
import hashlib
import numpy as np
from qwen_vl_utils import process_vision_info

def scene_cache_key(image, scene_id):
    """Deterministic key - an explicit scene_id """
    if scene_id:
        return hashlib.sha1(scene_id.encode()).hexdigest()
    h = hashlib.sha1()

    h.update(np.array(image.resize((64, 64))).tobytes())  #
    return h.hexdigest()


@torch.inference_mode()
def encode_scene(image , scene_id):
    """
    Run Qwen2-VL's vision model on image.
    """
    key = scene_cache_key(image, scene_id)

    cached = embed_cache.get(key) ## check if the image is already cached
    if cached is not None:

        return cached

    dummy_msgs  = build_qwen_messages(image, 'x')
    text_prompt = processor.apply_chat_template(
        dummy_msgs, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(dummy_msgs)


    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}


    embed_cache.set(key, inputs)


In [ ]:
@torch.inference_mode()
def ask_question(
    image,
    question,
    scene_id= None,
    max_new_tokens = 200,
) :

    t_total = time.perf_counter()


    cached_inputs = encode_scene(image, scene_id=scene_id)


    messages    = build_qwen_messages(image, question)
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}


    t_gen = time.perf_counter()
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    ## decode only the generated
    generated_ids = [
        out[len(inp):]
        for inp, out in zip(inputs['input_ids'], output_ids)
    ]

    answer = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return answer



In [ ]:
SCENE_ID = 'nuscenes_scene_0001'  ## scene number for different images
questions = [
    # Perception
    'Describe what each camera sees. List the key objects visible.',
    # Prediction
    'Are there any hazards or obstacles that require immediate attention?',
    # Planning
    'What action should the ego vehicle take next, and why?',
]

## sample image as input
base_img='/content/drive/MyDrive/data/v1.0-mini'
image_input = 'samples/CAM_FRONT/n008-2018-08-01-15-16-36-0400__CAM_FRONT__1533151609912404.jpg'
camera_image=Image.open((os.path.join(base_img,image_input)))
for i, q in enumerate(questions, 1):
    print(f'\n[Q{i}] {q}')
    answer = ask_question(camera_image, q, scene_id=SCENE_ID)
    print(f'[A{i}] {answer}')
    print('-' * 60)

